# LangGraph

LangGraph is a library for building stateful, multi-actor applications with LLMs. Think of it as a way to create complex AI workflows where different 'agents' or functions talk to each other.

It models your workflow as a **Graph**:
- **State**: The shared memory or data that is passed around.
- **Nodes**: The workers (Python functions or LLMs) that read the state, do something, and update the state.
- **Edges**: The rules or paths that decide which Node runs next.

In [1]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random


In [2]:
# Some useful constants

nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [ ]:
# Our favorite first step! Crew was doing this for us, by the way.
load_dotenv(override=True)

In [ ]:
def shout(text: Annotated[str, "something to be shouted"]) -> str:
    print(text.upper())
    return text.upper()

shout("hello")

## Steps in LangGraph

### Step 1: Define the State object

You can use any python object; but it's most common to use a TypedDict or a Pydantic BaseModel.

In [5]:

class State(BaseModel):
        
    messages: Annotated[list, add_messages]


### Step 2: Start the Graph Builder with this State class

In [6]:
graph_builder = StateGraph(State)

### 🏗️ Explaining Nodes

A Node is simply a Python function. When a Node is executed by LangGraph, it receives the current `State`. Its job is to return a dictionary or a new State object containing **updates** to the State. 

Because of the `add_messages` reducer we defined earlier, returning `messages: [new_message]` won't delete the old messages; it will append the new one!

### Step 3: Create a Node

A node can be any python function.

The reducer that we set before gets automatically called to combine this response with previous responses


In [ ]:
def our_first_node(old_state: State) -> State:

    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{"role": "assistant", "content": reply}]

    new_state = State(messages=messages)

    return new_state

graph_builder.add_node("first_node", our_first_node)

### 🛤️ Explaining Edges

Edges define the flow of execution. 
- `START` is a special node that indicates where the graph begins.
- `END` means the graph execution is finished.
- Here we just go START -> first_node -> END, which is a very simple linear flow. In real agents, you'd have loops and conditional edges!

### Step 4: Create Edges

In [ ]:
graph_builder.add_edge(START, "first_node")
graph_builder.add_edge("first_node", END)

### ⚙️ Compiling the Graph

Before you can run the graph, you must compile it. This turns your node-and-edge definitions into an executable application that can manage state, stream updates, and even pause/resume execution.

### Step 5: Compile the Graph

In [9]:
graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### That's it! Showtime!

In [ ]:
def chat(user_input: str, history):
    message = {"role": "user", "content": user_input}
    messages = [message]
    state = State(messages=messages)
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

### But why did I show you that?

To make the point that LangGraph is all about python functions - it doesn't need to involve LLMs!!

Now we'll do the 5 steps again, but in 1 shot:

In [12]:
# Step 1: Define the State object
class State(BaseModel):
    messages: Annotated[list, add_messages]


In [13]:
# Step 2: Start the Graph Builder with this State class
graph_builder = StateGraph(State)


In [ ]:
# Step 3: Create a Node

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

In [ ]:
# Step 4: Create Edges
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [ ]:
# Step 5: Compile the Graph
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### That's it! And, let's do this:

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content


gr.ChatInterface(chat, type="messages").launch()